# ***Alignment Score***

In [ ]:
# =========================
# Cell 1 — Install packages
# =========================

# Install required libraries
!pip -q install sentence-transformers pandas numpy scikit-learn matplotlib openpyxl

In [ ]:
# =========================
# Cell 2 — Imports
# =========================

# Core libraries
import os
import re
import json
from pathlib import Path

# Data libraries
import numpy as np
import pandas as pd

# Similarity computation
from sklearn.metrics.pairwise import cosine_similarity

# Optional plotting
import matplotlib.pyplot as plt

# Embedding model
from sentence_transformers import SentenceTransformer

# Hugging Face login
from huggingface_hub import login

In [ ]:
# =========================
# Cell 3 — Hugging Face login
# =========================

# Replace with your own token if needed
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

In [ ]:
# =========================
# Cell 4 — Configuration
# =========================

# Root directory containing all report folders
ROOT_DIR = "/content/drive/MyDrive/All Merged Results"

# Input CSV files expected inside each report folder
DESIGN_FILE = "design_testcases.csv"
REQUIREMENT_FILE = "requirement_testcases.csv"
CODE_FILE = "code_testcases.csv"

# Embedding model
# If T4 memory becomes a problem, switch to:
# MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = "BAAI/bge-large-en-v1.5"

# Output folder name inside each report folder
OUTPUT_DIR_NAME = "alignment_outputs"

# Master output folder at root
MASTER_OUTPUT_DIR = os.path.join(ROOT_DIR, "all_reports_alignment_summary")
os.makedirs(MASTER_OUTPUT_DIR, exist_ok=True)

# Optional controls
SAVE_HEATMAP = False
SAVE_SIM_MATRICES = True
NORMALIZE_TEXT = True

# BGE instruction prefix
USE_BGE_INSTRUCTION = True
BGE_INSTRUCTION = "Represent this test case for semantic similarity matching: "

# Heatmap display limits, only used if SAVE_HEATMAP = True
MAX_HEATMAP_ROWS = 80
MAX_HEATMAP_COLS = 120

In [ ]:
# =========================
# Cell 5 — Text normalization helper
# =========================

def normalize_text(text: str) -> str:
    """
    Normalize text before embedding:
    - handle missing values
    - strip spaces
    - collapse repeated whitespace
    - lowercase if enabled
    """
    text = "" if pd.isna(text) else str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    if NORMALIZE_TEXT:
        text = text.lower()
    return text

In [ ]:
# =========================
# Cell 6 — Build semantic text from CSV rows
# =========================

def combine_testcase_fields(row: pd.Series, source_name: str) -> str:
    """
    Build one semantic text representation for each testcase.
    This supports design, requirement, and code CSV header variations.
    """

    candidate_cols = {
        "test_case_id": [
            "test_case_id", "testcase_id", "tc_id",
            "Test Case ID"
        ],
        "title": [
            "title", "test_case_title", "name",
            "Title"
        ],
        "module": [
            "module", "module_name",
            "Module"
        ],
        "source_files_or_pages": [
            "source_files", "source_file", "file", "files",
            "Source UML Pages", "Source Pages"
        ],
        "covered_methods": [
            "covered_methods", "methods", "method", "covered_method"
        ],
        "preconditions": [
            "preconditions", "precondition",
            "Preconditions"
        ],
        "test_data": [
            "test_data", "data",
            "Test Data"
        ],
        "test_steps": [
            "test_steps", "steps", "procedure",
            "Test Steps"
        ],
        "expected_result": [
            "expected_result", "expected_results", "expected_output", "oracle",
            "Expected Result"
        ],
        "unit": [
            "unit"
        ],
        "project": [
            "project"
        ]
    }

    resolved = {}
    for key, options in candidate_cols.items():
        value = ""
        for c in options:
            if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
                value = str(row[c]).strip()
                break
        resolved[key] = value

    parts = [
        f"Source: {source_name}",
        f"Test Case ID: {resolved['test_case_id']}",
        f"Title: {resolved['title']}",
        f"Module: {resolved['module']}",
        f"Source Files or Pages: {resolved['source_files_or_pages']}",
        f"Covered Methods: {resolved['covered_methods']}",
        f"Preconditions: {resolved['preconditions']}",
        f"Test Data: {resolved['test_data']}",
        f"Test Steps: {resolved['test_steps']}",
        f"Expected Result: {resolved['expected_result']}",
        f"Unit: {resolved['unit']}",
        f"Project: {resolved['project']}",
    ]

    return normalize_text("\n".join(parts))

In [ ]:
# =========================
# Cell 7 — Read CSV files correctly
# =========================

def read_testcases_from_csv(file_path: str, source_name: str) -> pd.DataFrame:
    """
    Read testcase CSV and prepare:
    - tc_id
    - source label
    - embedding_text
    """
    df = pd.read_csv(file_path).copy().reset_index(drop=True)

    # Build semantic text used for embeddings
    df["embedding_text"] = df.apply(
        lambda row: combine_testcase_fields(row, source_name),
        axis=1
    )

    # Prefer existing testcase ids; otherwise generate simple ids
    if "test_case_id" in df.columns:
        df["tc_id"] = df["test_case_id"].astype(str)
    elif "testcase_id" in df.columns:
        df["tc_id"] = df["testcase_id"].astype(str)
    elif "Test Case ID" in df.columns:
        df["tc_id"] = df["Test Case ID"].astype(str)
    else:
        df["tc_id"] = [f"{source_name[:1].upper()}{i+1}" for i in range(len(df))]

    df["source"] = source_name
    return df

In [ ]:
# =========================
# Cell 8 — Embedding helpers
# =========================

def prepare_texts_for_embedding(texts, model_name):
    """
    Add instruction prefix when using BGE models.
    """
    if "bge" in model_name.lower() and USE_BGE_INSTRUCTION:
        return [BGE_INSTRUCTION + t for t in texts]
    return texts


def compute_embeddings(model, texts, model_name, batch_size=16):
    """
    Compute normalized embeddings for semantic similarity.
    """
    texts = prepare_texts_for_embedding(texts, model_name)
    embs = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embs

In [ ]:
# =========================
# Cell 9 — Best-match table
# =========================

def best_match_table(src_df, ref_df, sim_matrix):
    """
    For each source testcase, find the single best matching reference testcase.
    No threshold is used.
    """
    rows = []

    for i in range(sim_matrix.shape[0]):
        sims = sim_matrix[i]
        best_j = int(np.argmax(sims))
        best_score = float(sims[best_j])

        rows.append({
            "src_tc_id": src_df.iloc[i]["tc_id"],
            "best_ref_tc_id": ref_df.iloc[best_j]["tc_id"],
            "best_similarity": best_score,
            "src_embedding_text": src_df.iloc[i]["embedding_text"],
            "best_ref_embedding_text": ref_df.iloc[best_j]["embedding_text"]
        })

    return pd.DataFrame(rows)

In [ ]:
# =========================
# Cell 10 — Alignment score metric
# =========================

def alignment_metrics(match_df: pd.DataFrame):
    """
    Compute alignment score as the average of best cosine similarities.
    """
    total = len(match_df)
    alignment_score = float(match_df["best_similarity"].mean()) if total > 0 else 0.0

    return {
        "total_cases": total,
        "alignment_score": alignment_score
    }

In [ ]:
# =========================
# Cell 11 — Optional heatmap helper
# =========================

def plot_heatmap(sim_matrix, row_labels, col_labels, title, save_path=None):
    """
    Plot a truncated similarity heatmap for inspection.

    Note:
    This shows the full pairwise similarity matrix, which is often not very
    interpretable in this setup because many code testcases can be semantically similar.
    It is kept as an optional diagnostic tool only.
    """
    r = min(len(row_labels), MAX_HEATMAP_ROWS)
    c = min(len(col_labels), MAX_HEATMAP_COLS)

    sim_plot = sim_matrix[:r, :c]
    row_plot = row_labels[:r]
    col_plot = col_labels[:c]

    plt.figure(figsize=(max(8, c * 0.35), max(6, r * 0.25)))
    plt.imshow(sim_plot, aspect="auto")
    plt.colorbar(label="Cosine Similarity")
    plt.xticks(range(len(col_plot)), col_plot, rotation=90)
    plt.yticks(range(len(row_plot)), row_plot)
    plt.title(title)
    plt.xlabel("Code Test Cases")
    plt.ylabel("Source Test Cases")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
# =========================
# Cell 12 — Discover report folders
# =========================

def get_report_folders(root_dir: str):
    """
    Return all subfolders inside ROOT_DIR except the master summary folder.
    """
    folders = []
    master_name = os.path.basename(MASTER_OUTPUT_DIR)

    for name in sorted(os.listdir(root_dir)):
        full_path = os.path.join(root_dir, name)
        if os.path.isdir(full_path) and name != master_name:
            folders.append(full_path)

    return folders

In [ ]:
# =========================
# Cell 13 — Load embedding model
# =========================

# Load sentence-transformer model once
device = "cuda"
model = SentenceTransformer(MODEL_NAME, device=device)
print("Loaded model:", MODEL_NAME)

In [ ]:
# =========================
# Cell 14 — Process all report folders
# =========================

report_dirs = get_report_folders(ROOT_DIR)
print("Total report folders found:", len(report_dirs))

all_summary_rows = []
skipped_reports = []

for report_dir in report_dirs:
    report_name = os.path.basename(report_dir)
    print(f"\nProcessing: {report_name}")

    output_dir = os.path.join(report_dir, OUTPUT_DIR_NAME)
    os.makedirs(output_dir, exist_ok=True)

    design_path = os.path.join(report_dir, DESIGN_FILE)
    requirement_path = os.path.join(report_dir, REQUIREMENT_FILE)
    code_path = os.path.join(report_dir, CODE_FILE)

    # Check required files
    missing_files = []
    for p in [design_path, requirement_path, code_path]:
        if not os.path.exists(p):
            missing_files.append(os.path.basename(p))

    if len(missing_files) > 0:
        skipped_reports.append({
            "report_folder": report_name,
            "reason": f"Missing files: {', '.join(missing_files)}"
        })
        print(f"Skipped {report_name} -> Missing files")
        continue

    try:
        # Load CSV files
        design_df = read_testcases_from_csv(design_path, "design")
        requirement_df = read_testcases_from_csv(requirement_path, "requirement")
        code_df = read_testcases_from_csv(code_path, "code")

        # Optional verification for first 1-2 rows only if needed
        # print(design_df["embedding_text"].head(1).tolist())

        # Compute embeddings
        design_emb = compute_embeddings(
            model,
            design_df["embedding_text"].tolist(),
            MODEL_NAME,
            batch_size=16
        )

        requirement_emb = compute_embeddings(
            model,
            requirement_df["embedding_text"].tolist(),
            MODEL_NAME,
            batch_size=16
        )

        code_emb = compute_embeddings(
            model,
            code_df["embedding_text"].tolist(),
            MODEL_NAME,
            batch_size=16
        )

        # -------------------------
        # Design to Code alignment
        # -------------------------
        sim_design_code = cosine_similarity(design_emb, code_emb)

        design_code_matches = best_match_table(
            src_df=design_df,
            ref_df=code_df,
            sim_matrix=sim_design_code
        )

        design_code_metrics = alignment_metrics(design_code_matches)

        design_code_matches.to_csv(
            os.path.join(output_dir, "design_to_code_best_matches.csv"),
            index=False
        )

        # Optional similarity heatmap
        if SAVE_HEATMAP:
            plot_heatmap(
                sim_design_code,
                row_labels=design_df["tc_id"].tolist(),
                col_labels=code_df["tc_id"].tolist(),
                title=f"{report_name} - Design to Code Similarity Heatmap",
                save_path=os.path.join(output_dir, "design_to_code_heatmap.png")
            )

        # ------------------------------
        # Requirement to Code alignment
        # ------------------------------
        sim_req_code = cosine_similarity(requirement_emb, code_emb)

        req_code_matches = best_match_table(
            src_df=requirement_df,
            ref_df=code_df,
            sim_matrix=sim_req_code
        )

        req_code_metrics = alignment_metrics(req_code_matches)

        req_code_matches.to_csv(
            os.path.join(output_dir, "requirement_to_code_best_matches.csv"),
            index=False
        )

        # Optional similarity heatmap
        if SAVE_HEATMAP:
            plot_heatmap(
                sim_req_code,
                row_labels=requirement_df["tc_id"].tolist(),
                col_labels=code_df["tc_id"].tolist(),
                title=f"{report_name} - Requirement to Code Similarity Heatmap",
                save_path=os.path.join(output_dir, "requirement_to_code_heatmap.png")
            )

        # -------------------------
        # Save per-report summary
        # -------------------------
        summary = {
            "report_folder": report_name,
            "model_name": MODEL_NAME,
            "design_total_source_cases": design_code_metrics["total_cases"],
            "design_to_code_alignment": design_code_metrics["alignment_score"],
            "requirement_total_source_cases": req_code_metrics["total_cases"],
            "requirement_to_code_alignment": req_code_metrics["alignment_score"]
        }

        with open(os.path.join(output_dir, "alignment_summary.json"), "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)

        final_df = pd.DataFrame([
            {
                "alignment_type": "design_to_code",
                "model_name": MODEL_NAME,
                "total_source_cases": design_code_metrics["total_cases"],
                "alignment_score": design_code_metrics["alignment_score"]
            },
            {
                "alignment_type": "requirement_to_code",
                "model_name": MODEL_NAME,
                "total_source_cases": req_code_metrics["total_cases"],
                "alignment_score": req_code_metrics["alignment_score"]
            }
        ])

        final_df.to_csv(os.path.join(output_dir, "final_alignment_scores.csv"), index=False)
        final_df.to_excel(os.path.join(output_dir, "final_alignment_scores.xlsx"), index=False)

        # Save full similarity matrices if enabled
        if SAVE_SIM_MATRICES:
            design_code_sim_df = pd.DataFrame(
                sim_design_code,
                index=design_df["tc_id"].tolist(),
                columns=code_df["tc_id"].tolist()
            )
            design_code_sim_df.to_csv(
                os.path.join(output_dir, "design_to_code_similarity_matrix.csv")
            )

            req_code_sim_df = pd.DataFrame(
                sim_req_code,
                index=requirement_df["tc_id"].tolist(),
                columns=code_df["tc_id"].tolist()
            )
            req_code_sim_df.to_csv(
                os.path.join(output_dir, "requirement_to_code_similarity_matrix.csv")
            )

        # Add to master summary
        all_summary_rows.append(summary)

        print(
            f"Done: {report_name} | "
            f"Design->Code={design_code_metrics['alignment_score']:.4f} | "
            f"Requirement->Code={req_code_metrics['alignment_score']:.4f}"
        )

    except Exception as e:
        skipped_reports.append({
            "report_folder": report_name,
            "reason": str(e)
        })
        print(f"Skipped {report_name} -> Error: {e}")

In [ ]:
# =========================
# Cell 15 — Save master summary across all reports
# =========================

master_summary_df = pd.DataFrame(all_summary_rows)

display(master_summary_df.head() if len(master_summary_df) > 0 else pd.DataFrame())

master_summary_df.to_csv(
    os.path.join(MASTER_OUTPUT_DIR, "all_reports_alignment_summary.csv"),
    index=False
)

master_summary_df.to_excel(
    os.path.join(MASTER_OUTPUT_DIR, "all_reports_alignment_summary.xlsx"),
    index=False
)

print("Saved master alignment summary.")

In [ ]:
# =========================
# Cell 16 — Save skipped report log
# =========================

skipped_df = pd.DataFrame(skipped_reports)

display(skipped_df.head() if len(skipped_df) > 0 else pd.DataFrame())

skipped_df.to_csv(
    os.path.join(MASTER_OUTPUT_DIR, "skipped_reports_log.csv"),
    index=False
)

print("Saved skipped reports log.")

In [ ]:
# =========================
# Cell 17 — Aggregate summary statistics
# =========================

if len(master_summary_df) > 0:
    aggregate_stats = {
        "num_reports_processed": int(len(master_summary_df)),
        "avg_design_to_code_alignment": float(master_summary_df["design_to_code_alignment"].mean()),
        "avg_requirement_to_code_alignment": float(master_summary_df["requirement_to_code_alignment"].mean())
    }

    print(json.dumps(aggregate_stats, indent=2))

    with open(
        os.path.join(MASTER_OUTPUT_DIR, "aggregate_alignment_stats.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(aggregate_stats, f, indent=2)
else:
    print("No reports processed.")

In [ ]:
# =========================
# Cell 18 — Optional report-wise plot
# =========================

if len(master_summary_df) > 0:
    plot_df = master_summary_df.copy()

    plt.figure(figsize=(14, 6))
    plt.plot(
        plot_df["report_folder"],
        plot_df["design_to_code_alignment"],
        marker="o",
        label="Design to Code"
    )
    plt.plot(
        plot_df["report_folder"],
        plot_df["requirement_to_code_alignment"],
        marker="o",
        label="Requirement to Code"
    )
    plt.xticks(rotation=90)
    plt.ylabel("Alignment Score")
    plt.title("Report-wise Alignment Score")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# =========================
# Cell 19 — Optional: combined best-match files across all reports
# =========================

# This cell combines all per-report best-match CSVs into two master files:
# one for design-to-code and one for requirement-to-code

combined_design_rows = []
combined_requirement_rows = []

for report_dir in report_dirs:
    report_name = os.path.basename(report_dir)
    output_dir = os.path.join(report_dir, OUTPUT_DIR_NAME)

    design_match_path = os.path.join(output_dir, "design_to_code_best_matches.csv")
    requirement_match_path = os.path.join(output_dir, "requirement_to_code_best_matches.csv")

    if os.path.exists(design_match_path):
        temp_df = pd.read_csv(design_match_path)
        temp_df.insert(0, "report_folder", report_name)
        combined_design_rows.append(temp_df)

    if os.path.exists(requirement_match_path):
        temp_df = pd.read_csv(requirement_match_path)
        temp_df.insert(0, "report_folder", report_name)
        combined_requirement_rows.append(temp_df)

if len(combined_design_rows) > 0:
    combined_design_df = pd.concat(combined_design_rows, ignore_index=True)
    combined_design_df.to_csv(
        os.path.join(MASTER_OUTPUT_DIR, "all_reports_design_to_code_best_matches.csv"),
        index=False
    )
    print("Saved combined design-to-code best matches.")

if len(combined_requirement_rows) > 0:
    combined_requirement_df = pd.concat(combined_requirement_rows, ignore_index=True)
    combined_requirement_df.to_csv(
        os.path.join(MASTER_OUTPUT_DIR, "all_reports_requirement_to_code_best_matches.csv"),
        index=False
    )
    print("Saved combined requirement-to-code best matches.")

# Semantic Coverage

In [ ]:
# =========================
# Cell 1 — Install packages
# =========================

# Install required libraries
!pip -q install sentence-transformers pandas numpy scikit-learn openpyxl

# =========================
# Cell 2 — Imports
# =========================

# Core libraries
import os
import re
import json
from pathlib import Path

# Data libraries
import numpy as np
import pandas as pd

# Similarity computation
from sklearn.metrics.pairwise import cosine_similarity

# Embedding model
from sentence_transformers import SentenceTransformer

# Hugging Face login
from huggingface_hub import login

# =========================
# Cell 3 — Hugging Face login
# =========================

# Replace with your own token if needed
# HuggingFace token access
login(token="YOUR_HF_TOKEN")

# =========================
# Cell 4 — Configuration
# =========================

# Root directory containing all cleaned report folders
ROOT_DIR = "/content/drive/MyDrive/SEMANTIC_COVERAGE_READY/mistral 7b/qwen"

# Output folder inside each report folder
OUTPUT_DIR_NAME = "semantic_coverage_outputs"

# Master output folder at root
MASTER_OUTPUT_DIR = os.path.join(ROOT_DIR, "all_reports_semantic_coverage_summary")
os.makedirs(MASTER_OUTPUT_DIR, exist_ok=True)

# Embedding model
# If T4 memory becomes a problem, switch to:
# MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = "BAAI/bge-large-en-v1.5"

# Text normalization
NORMALIZE_TEXT = True

# BGE instruction prefix
USE_BGE_INSTRUCTION = True
BGE_INSTRUCTION = "Represent this software testing text for semantic similarity matching: "

# Similarity search transparency
# None means save similarity against ALL testcases per extracted item
# Example: set to 10 if you only want top-10 ranked matches saved
SIMILARITY_SEARCH_TOP_K = None

# =========================
# Cell 5 — Text normalization helpers
# =========================

def normalize_text(text: str) -> str:
    """
    Normalize text:
    - handle missing values
    - strip spaces
    - collapse repeated whitespace
    - lowercase if enabled
    """
    text = "" if text is None or pd.isna(text) else str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    if NORMALIZE_TEXT:
        text = text.lower()
    return text


def unique_keep_order(items):
    """
    Remove duplicates while preserving order.
    """
    seen = set()
    out = []
    for x in items:
        x = normalize_text(x)
        if x and x != "none" and x not in seen:
            seen.add(x)
            out.append(x)
    return out

# =========================
# Cell 6 — File discovery helpers
# =========================

def get_report_folders(root_dir: str):
    """
    Return all actual report folders inside ROOT_DIR,
    excluding the master summary folder.
    """
    folders = []
    root = Path(root_dir)
    master_name = os.path.basename(MASTER_OUTPUT_DIR)

    for item in sorted(root.iterdir(), key=lambda x: x.name.lower()):
        if item.is_dir() and item.name != master_name:
            folders.append(str(item))

    return folders


def get_report_input_pair(report_dir: str):
    """
    Select exactly one .txt and one .csv from each report folder.

    Expected cleaned structure:
      Report_X_uml_pages/
        - Report_X_uml_pages.txt
        - Report_X_uml_pages_testcases.csv

    Extra files, if present, are ignored safely.
    """
    txt_files = []
    csv_files = []

    for item in sorted(Path(report_dir).iterdir(), key=lambda x: x.name.lower()):
        if not item.is_file():
            continue

        lower_name = item.name.lower()

        if lower_name.endswith(".txt"):
            txt_files.append(str(item))

        elif lower_name.endswith(".csv"):
            # Exclude generated output files in case of re-run
            if "semantic_coverage" in lower_name:
                continue
            if "entity_coverage" in lower_name:
                continue
            if "flow_coverage" in lower_name:
                continue
            if "conditional_coverage" in lower_name:
                continue
            if "similarity_search" in lower_name:
                continue
            if lower_name == "summary.csv":
                continue
            csv_files.append(str(item))

    # Prefer report txt
    txt_ranked = sorted(
        txt_files,
        key=lambda x: (
            0 if "report_" in os.path.basename(x).lower() and "uml_pages" in os.path.basename(x).lower() else 1,
            os.path.basename(x).lower()
        )
    )

    # Prefer testcase csv
    csv_ranked = sorted(
        csv_files,
        key=lambda x: (
            0 if ("uml_pages" in os.path.basename(x).lower() and "testcases" in os.path.basename(x).lower()) else
            1 if "testcases" in os.path.basename(x).lower() else
            2,
            os.path.basename(x).lower()
        )
    )

    selected_txt = txt_ranked[0] if len(txt_ranked) > 0 else None
    selected_csv = csv_ranked[0] if len(csv_ranked) > 0 else None

    return {
        "txt_path": selected_txt,
        "csv_path": selected_csv,
        "txt_files_found": txt_files,
        "csv_files_found": csv_files
    }

# =========================
# Cell 7 — Split report text into pages
# =========================

def split_pages(raw_text: str):
    """
    Split UML description text into pages using markers like:
    ===== Report_x.pdf | Page y =====
    """
    pattern = r"=+\s*(.*?)\s*\|\s*Page\s+(\d+)\s*=+"
    matches = list(re.finditer(pattern, raw_text))

    pages = []

    for i, m in enumerate(matches):
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(raw_text)

        pages.append({
            "report_name": m.group(1).strip(),
            "page_no": int(m.group(2)),
            "content": raw_text[start:end].strip()
        })

    return pages

# =========================
# Cell 8 — Extract section blocks from each page
# =========================

SECTION_HEADERS = [
    "UML Type",
    "Top Components",
    "Components",
    "Actors / Lifelines",
    "Relationships / Links",
    "Main Flows (step by step)",
    "Main Flows",
    "Conditions / Branches",
    "Loops / Repetitions",
    "Notes / Annotations",
    "Observations"
]

def extract_sections(page_text: str):
    """
    Extract named sections from a page block.
    """
    lines = page_text.splitlines()
    sections = {}
    current = None
    buffer = []

    def flush():
        nonlocal current, buffer
        if current is not None:
            sections[current] = "\n".join(buffer).strip()
        buffer = []

    for line in lines:
        line_stripped = line.strip()
        matched_header = None

        for h in SECTION_HEADERS:
            if line_stripped.startswith(h + ":"):
                matched_header = h
                break

        if matched_header:
            flush()
            current = matched_header
            content_after_colon = line_stripped[len(matched_header) + 1:].strip()
            buffer = [content_after_colon] if content_after_colon else []
        else:
            if current is not None:
                buffer.append(line)

    flush()
    return sections

# =========================
# Cell 9 — Generic extraction of entities, flows, and conditionals
# =========================

def extract_entities_from_section(section_text: str):
    """
    Extract entity-like items from Components / Actors / Lifelines / Top Components.
    """
    items = []
    if not section_text.strip():
        return items

    # Handle comma-separated top components
    if "," in section_text and "\n" not in section_text:
        for part in section_text.split(","):
            part = part.strip()
            if part:
                items.append(part)

    # Handle bullet-based sections
    for line in section_text.splitlines():
        line = line.strip()
        if not line or line.lower() == "none":
            continue

        line = re.sub(r"^-\s*", "", line).strip()

        # Keep the main object before metadata
        m = re.match(r"^([^—\-:;]+)", line)
        if m:
            items.append(m.group(1).strip())
        else:
            items.append(line)

    return unique_keep_order(items)


def extract_flows_from_section(section_text: str):
    """
    Extract flow-like items from Relationships / Links, Main Flows, Observations.
    """
    items = []
    if not section_text.strip():
        return items

    for line in section_text.splitlines():
        line = line.strip()
        if not line or line.lower() == "none":
            continue

        line = re.sub(r"^\d+\.\s*", "", line)
        line = re.sub(r"^-\s*", "", line).strip()

        if line:
            items.append(line)

    return unique_keep_order(items)


def extract_conditionals_from_section(section_text: str):
    """
    Extract conditionals from Conditions / Branches.
    """
    items = []
    if not section_text.strip():
        return items

    for line in section_text.splitlines():
        line = line.strip()
        if not line or line.lower() == "none":
            continue

        line = re.sub(r"^\d+\.\s*", "", line)
        line = re.sub(r"^-\s*", "", line).strip()

        if line:
            items.append(line)

    return unique_keep_order(items)

# =========================
# Cell 10 — Build semantic inventory from UML text
# =========================

def build_semantic_inventory(txt_path: str):
    """
    Build report-level semantic inventory:
    - entities
    - flows
    - conditionals
    """
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

    pages = split_pages(raw_text)

    all_entities = []
    all_flows = []
    all_conditionals = []
    page_records = []

    for page in pages:
        sections = extract_sections(page["content"])

        page_entities = []
        page_flows = []
        page_conditionals = []

        if "Top Components" in sections:
            page_entities.extend(extract_entities_from_section(sections["Top Components"]))

        if "Components" in sections:
            page_entities.extend(extract_entities_from_section(sections["Components"]))

        if "Actors / Lifelines" in sections:
            page_entities.extend(extract_entities_from_section(sections["Actors / Lifelines"]))

        if "Relationships / Links" in sections:
            page_flows.extend(extract_flows_from_section(sections["Relationships / Links"]))

        if "Main Flows (step by step)" in sections:
            page_flows.extend(extract_flows_from_section(sections["Main Flows (step by step)"]))

        if "Main Flows" in sections:
            page_flows.extend(extract_flows_from_section(sections["Main Flows"]))

        if "Observations" in sections:
            page_flows.extend(extract_flows_from_section(sections["Observations"]))

        if "Conditions / Branches" in sections:
            page_conditionals.extend(extract_conditionals_from_section(sections["Conditions / Branches"]))

        page_entities = unique_keep_order(page_entities)
        page_flows = unique_keep_order(page_flows)
        page_conditionals = unique_keep_order(page_conditionals)

        all_entities.extend(page_entities)
        all_flows.extend(page_flows)
        all_conditionals.extend(page_conditionals)

        page_records.append({
            "page_no": page["page_no"],
            "uml_type": sections.get("UML Type", ""),
            "entities": page_entities,
            "flows": page_flows,
            "conditionals": page_conditionals
        })

    inventory = {
        "entities": unique_keep_order(all_entities),
        "flows": unique_keep_order(all_flows),
        "conditionals": unique_keep_order(all_conditionals),
        "pages": page_records
    }

    return inventory

# =========================
# Cell 11 — Build semantic text from generated testcase CSV rows
# =========================

def build_testcase_semantic_text(row: pd.Series) -> str:
    """
    Build rich semantic text from generated testcase CSV fields.
    Supports common design/requirement generated testcase headers.
    """
    candidate_cols = {
        "test_case_id": [
            "test_case_id", "testcase_id", "tc_id",
            "Test Case ID"
        ],
        "title": [
            "title", "test_case_title", "name",
            "Title"
        ],
        "module": [
            "module", "module_name",
            "Module"
        ],
        "source_pages": [
            "source_uml_pages", "source uml pages", "source_pages",
            "Source UML Pages", "Source Pages"
        ],
        "preconditions": [
            "preconditions", "precondition",
            "Preconditions"
        ],
        "test_data": [
            "test_data", "data",
            "Test Data"
        ],
        "test_steps": [
            "test_steps", "steps", "procedure",
            "Test Steps"
        ],
        "expected_result": [
            "expected_result", "expected_results", "expected_output", "oracle",
            "Expected Result"
        ]
    }

    resolved = {}
    for key, options in candidate_cols.items():
        value = ""
        for c in options:
            if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
                value = str(row[c]).strip()
                break
        resolved[key] = value

    parts = [
        f"Test Case ID: {resolved['test_case_id']}",
        f"Title: {resolved['title']}",
        f"Module: {resolved['module']}",
        f"Source Pages: {resolved['source_pages']}",
        f"Preconditions: {resolved['preconditions']}",
        f"Test Data: {resolved['test_data']}",
        f"Test Steps: {resolved['test_steps']}",
        f"Expected Result: {resolved['expected_result']}"
    ]

    return normalize_text("\n".join(parts))

# =========================
# Cell 12 — Read generated testcase CSV
# =========================

def read_generated_testcases_csv(file_path: str) -> pd.DataFrame:
    """
    Read generated testcase CSV and prepare:
    - tc_id
    - semantic_text
    """
    df = pd.read_csv(file_path).copy().reset_index(drop=True)

    if "test_case_id" in df.columns:
        df["tc_id"] = df["test_case_id"].astype(str)
    elif "testcase_id" in df.columns:
        df["tc_id"] = df["testcase_id"].astype(str)
    elif "Test Case ID" in df.columns:
        df["tc_id"] = df["Test Case ID"].astype(str)
    else:
        df["tc_id"] = [f"TC_{i+1}" for i in range(len(df))]

    df["semantic_text"] = df.apply(build_testcase_semantic_text, axis=1)
    return df

# =========================
# Cell 13 — Embedding helpers
# =========================

def prepare_texts_for_embedding(texts, model_name):
    """
    Add BGE instruction prefix if needed.
    """
    if "bge" in model_name.lower() and USE_BGE_INSTRUCTION:
        return [BGE_INSTRUCTION + t for t in texts]
    return texts


def encode_texts(model, texts, model_name, batch_size=16):
    """
    Compute normalized embeddings.
    """
    texts = prepare_texts_for_embedding(texts, model_name)
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embeddings

# =========================
# Cell 14 — Similarity search tables + best-match tables
# =========================

def similarity_search_table(items, item_type, tc_df, testcase_embeddings, model, model_name, top_k=None):
    """
    Save ranked similarity search results for each extracted item
    against testcase rows.

    This is for transparency/reviewer understanding.
    Final coverage still uses only rank 1 (best similarity).
    """
    rows = []

    if len(items) == 0:
        return pd.DataFrame(columns=[
            "item_type",
            "item_text",
            "rank",
            "matched_testcase_id",
            "similarity_score",
            "matched_testcase_text"
        ])

    item_embeddings = encode_texts(model, items, model_name, batch_size=16)
    sim_matrix = cosine_similarity(item_embeddings, testcase_embeddings)

    num_testcases = len(tc_df)
    k = num_testcases if top_k is None else min(top_k, num_testcases)

    for i, item_text in enumerate(items):
        sims = sim_matrix[i]
        ranked_idx = np.argsort(-sims)[:k]

        for rank_pos, j in enumerate(ranked_idx, start=1):
            rows.append({
                "item_type": item_type,
                "item_text": item_text,
                "rank": rank_pos,
                "matched_testcase_id": tc_df.iloc[j]["tc_id"],
                "similarity_score": float(sims[j]),
                "matched_testcase_text": tc_df.iloc[j]["semantic_text"]
            })

    return pd.DataFrame(rows)


def best_similarity_table(items, item_type, tc_df, testcase_embeddings, model, model_name):
    """
    For each semantic item (entity/flow/conditional),
    find the most similar generated testcase using cosine similarity.
    No threshold is used.
    """
    rows = []

    if len(items) == 0:
        return pd.DataFrame(columns=[
            "item_type", "item_text", "best_testcase_id",
            "best_similarity", "best_testcase_text"
        ])

    item_embeddings = encode_texts(model, items, model_name, batch_size=16)
    sim_matrix = cosine_similarity(item_embeddings, testcase_embeddings)

    for i, item_text in enumerate(items):
        sims = sim_matrix[i]
        best_j = int(np.argmax(sims))
        best_score = float(sims[best_j])

        rows.append({
            "item_type": item_type,
            "item_text": item_text,
            "best_testcase_id": tc_df.iloc[best_j]["tc_id"],
            "best_similarity": best_score,
            "best_testcase_text": tc_df.iloc[best_j]["semantic_text"]
        })

    return pd.DataFrame(rows)

# =========================
# Cell 15 — Load embedding model
# =========================

# Load sentence-transformer model once
device = "cuda"
model = SentenceTransformer(MODEL_NAME, device=device)
print("Loaded model:", MODEL_NAME)

# =========================
# Cell 16 — Process all report folders
# =========================

report_dirs = get_report_folders(ROOT_DIR)
print("Total report folders found:", len(report_dirs))

all_summary_rows = []
skipped_reports = []

for report_dir in report_dirs:
    report_name = os.path.basename(report_dir)
    print(f"\nProcessing: {report_name}")

    output_dir = os.path.join(report_dir, OUTPUT_DIR_NAME)
    os.makedirs(output_dir, exist_ok=True)

    pair_info = get_report_input_pair(report_dir)
    uml_txt_path = pair_info["txt_path"]
    testcase_csv_path = pair_info["csv_path"]

    if uml_txt_path is None or testcase_csv_path is None:
        reason_parts = []
        if uml_txt_path is None:
            reason_parts.append(f"UML description txt not found | txt_found={len(pair_info['txt_files_found'])}")
        if testcase_csv_path is None:
            reason_parts.append(f"generated testcase csv not found | csv_found={len(pair_info['csv_files_found'])}")

        skipped_reports.append({
            "report_folder": report_name,
            "reason": "; ".join(reason_parts),
            "txt_candidates": " | ".join([os.path.basename(x) for x in pair_info["txt_files_found"]]),
            "csv_candidates": " | ".join([os.path.basename(x) for x in pair_info["csv_files_found"]])
        })
        print(f"Skipped {report_name} -> {'; '.join(reason_parts)}")
        continue

    try:
        # Build semantic inventory from UML description
        inventory = build_semantic_inventory(uml_txt_path)

        # Save semantic inventory for that report
        with open(os.path.join(output_dir, "semantic_inventory.json"), "w", encoding="utf-8") as f:
            json.dump(inventory, f, indent=2, ensure_ascii=False)

        pd.DataFrame({"entity": inventory["entities"]}).to_csv(
            os.path.join(output_dir, "entities.csv"), index=False
        )
        pd.DataFrame({"flow": inventory["flows"]}).to_csv(
            os.path.join(output_dir, "flows.csv"), index=False
        )
        pd.DataFrame({"conditional": inventory["conditionals"]}).to_csv(
            os.path.join(output_dir, "conditionals.csv"), index=False
        )

        # Load generated testcase CSV
        tc_df = read_generated_testcases_csv(testcase_csv_path)

        # Compute testcase embeddings
        testcase_embeddings = encode_texts(
            model,
            tc_df["semantic_text"].tolist(),
            MODEL_NAME,
            batch_size=16
        )

        # -------------------------
        # Entity similarity search + coverage
        # -------------------------
        entity_similarity_search_df = similarity_search_table(
            items=inventory["entities"],
            item_type="entity",
            tc_df=tc_df,
            testcase_embeddings=testcase_embeddings,
            model=model,
            model_name=MODEL_NAME,
            top_k=SIMILARITY_SEARCH_TOP_K
        )

        entity_df = best_similarity_table(
            items=inventory["entities"],
            item_type="entity",
            tc_df=tc_df,
            testcase_embeddings=testcase_embeddings,
            model=model,
            model_name=MODEL_NAME
        )
        entity_coverage = float(entity_df["best_similarity"].mean()) if len(entity_df) > 0 else 0.0

        # -------------------------
        # Flow similarity search + coverage
        # -------------------------
        flow_similarity_search_df = similarity_search_table(
            items=inventory["flows"],
            item_type="flow",
            tc_df=tc_df,
            testcase_embeddings=testcase_embeddings,
            model=model,
            model_name=MODEL_NAME,
            top_k=SIMILARITY_SEARCH_TOP_K
        )

        flow_df = best_similarity_table(
            items=inventory["flows"],
            item_type="flow",
            tc_df=tc_df,
            testcase_embeddings=testcase_embeddings,
            model=model,
            model_name=MODEL_NAME
        )
        flow_coverage = float(flow_df["best_similarity"].mean()) if len(flow_df) > 0 else 0.0

        # -------------------------
        # Conditional similarity search + coverage
        # Rule:
        # if no conditionals exist in UML text,
        # treat conditional coverage as 1.0
        # -------------------------
        if len(inventory["conditionals"]) == 0:
            conditional_similarity_search_df = pd.DataFrame(columns=[
                "item_type",
                "item_text",
                "rank",
                "matched_testcase_id",
                "similarity_score",
                "matched_testcase_text"
            ])
            conditional_df = pd.DataFrame(columns=[
                "item_type", "item_text", "best_testcase_id",
                "best_similarity", "best_testcase_text"
            ])
            conditional_coverage = 1.0
            conditional_status = "no_conditionals_in_txt_assumed_100_percent"
        else:
            conditional_similarity_search_df = similarity_search_table(
                items=inventory["conditionals"],
                item_type="conditional",
                tc_df=tc_df,
                testcase_embeddings=testcase_embeddings,
                model=model,
                model_name=MODEL_NAME,
                top_k=SIMILARITY_SEARCH_TOP_K
            )

            conditional_df = best_similarity_table(
                items=inventory["conditionals"],
                item_type="conditional",
                tc_df=tc_df,
                testcase_embeddings=testcase_embeddings,
                model=model,
                model_name=MODEL_NAME
            )
            conditional_coverage = float(conditional_df["best_similarity"].mean()) if len(conditional_df) > 0 else 0.0
            conditional_status = "computed_from_txt_conditionals"

        # -------------------------
        # Final semantic coverage
        # -------------------------
        semantic_coverage = (
            entity_coverage + flow_coverage + conditional_coverage
        ) / 3.0

        # -------------------------
        # Save similarity search files (full ranked search transparency)
        # -------------------------
        entity_similarity_search_df.to_csv(
            os.path.join(output_dir, "entity_similarity_search.csv"),
            index=False
        )
        flow_similarity_search_df.to_csv(
            os.path.join(output_dir, "flow_similarity_search.csv"),
            index=False
        )
        conditional_similarity_search_df.to_csv(
            os.path.join(output_dir, "conditional_similarity_search.csv"),
            index=False
        )

        # -------------------------
        # Save best-match detail files (used for coverage calculation)
        # -------------------------
        entity_df.to_csv(os.path.join(output_dir, "entity_coverage_details.csv"), index=False)
        flow_df.to_csv(os.path.join(output_dir, "flow_coverage_details.csv"), index=False)
        conditional_df.to_csv(os.path.join(output_dir, "conditional_coverage_details.csv"), index=False)

        # Save summary table for that report
        summary_df = pd.DataFrame([
            {"metric": "EntityCoverage", "value": entity_coverage},
            {"metric": "FlowCoverage", "value": flow_coverage},
            {"metric": "ConditionalCoverage", "value": conditional_coverage},
            {"metric": "SemanticCoverage", "value": semantic_coverage}
        ])

        summary_df.to_csv(os.path.join(output_dir, "semantic_coverage_summary.csv"), index=False)
        summary_df.to_excel(os.path.join(output_dir, "semantic_coverage_summary.xlsx"), index=False)

        # Save an explanation JSON for reviewer transparency
        similarity_explanation = {
            "report_folder": report_name,
            "similarity_search_saved": True,
            "similarity_search_top_k": SIMILARITY_SEARCH_TOP_K,
            "coverage_logic": {
                "entity_coverage": "mean of best similarity values across extracted entities",
                "flow_coverage": "mean of best similarity values across extracted flows",
                "conditional_coverage": "mean of best similarity values across extracted conditionals; if no conditionals exist in txt, conditional coverage is set to 1.0",
                "semantic_coverage": "average of entity_coverage, flow_coverage, and conditional_coverage"
            },
            "files_saved": [
                "entity_similarity_search.csv",
                "flow_similarity_search.csv",
                "conditional_similarity_search.csv",
                "entity_coverage_details.csv",
                "flow_coverage_details.csv",
                "conditional_coverage_details.csv",
                "semantic_coverage_summary.csv",
                "semantic_inventory.json"
            ]
        }

        with open(os.path.join(output_dir, "similarity_search_explanation.json"), "w", encoding="utf-8") as f:
            json.dump(similarity_explanation, f, indent=2)

        report_summary = {
            "report_folder": report_name,
            "model_name": MODEL_NAME,
            "uml_txt_file": os.path.basename(uml_txt_path),
            "testcase_csv_file": os.path.basename(testcase_csv_path),
            "num_testcases": int(len(tc_df)),
            "num_entities": int(len(inventory["entities"])),
            "num_flows": int(len(inventory["flows"])),
            "num_conditionals": int(len(inventory["conditionals"])),
            "entity_coverage": float(entity_coverage),
            "flow_coverage": float(flow_coverage),
            "conditional_coverage": float(conditional_coverage),
            "semantic_coverage": float(semantic_coverage),
            "conditional_status": conditional_status
        }

        with open(os.path.join(output_dir, "semantic_coverage_summary.json"), "w", encoding="utf-8") as f:
            json.dump(report_summary, f, indent=2)

        all_summary_rows.append(report_summary)

        print(
            f"Done: {report_name} | "
            f"TXT={os.path.basename(uml_txt_path)} | "
            f"CSV={os.path.basename(testcase_csv_path)} | "
            f"Entity={entity_coverage:.4f} | "
            f"Flow={flow_coverage:.4f} | "
            f"Conditional={conditional_coverage:.4f} | "
            f"Semantic={semantic_coverage:.4f} | "
            f"n_entities={len(inventory['entities'])} | "
            f"n_flows={len(inventory['flows'])} | "
            f"n_conditionals={len(inventory['conditionals'])}"
        )

    except Exception as e:
        skipped_reports.append({
            "report_folder": report_name,
            "reason": str(e),
            "txt_candidates": " | ".join([os.path.basename(x) for x in pair_info["txt_files_found"]]),
            "csv_candidates": " | ".join([os.path.basename(x) for x in pair_info["csv_files_found"]])
        })
        print(f"Skipped {report_name} -> Error: {e}")

# =========================
# Cell 17 — Save master summary across all reports
# =========================

master_summary_df = pd.DataFrame(all_summary_rows)

display(master_summary_df.head() if len(master_summary_df) > 0 else pd.DataFrame())

master_summary_df.to_csv(
    os.path.join(MASTER_OUTPUT_DIR, "all_reports_semantic_coverage_summary.csv"),
    index=False
)

master_summary_df.to_excel(
    os.path.join(MASTER_OUTPUT_DIR, "all_reports_semantic_coverage_summary.xlsx"),
    index=False
)

print("Saved master semantic coverage summary.")

# =========================
# Cell 18 — Save skipped report log
# =========================

skipped_df = pd.DataFrame(skipped_reports)

display(skipped_df.head() if len(skipped_df) > 0 else pd.DataFrame())

skipped_df.to_csv(
    os.path.join(MASTER_OUTPUT_DIR, "skipped_reports_log.csv"),
    index=False
)

print("Saved skipped reports log.")

# =========================
# Cell 19 — Aggregate summary statistics
# =========================

if len(master_summary_df) > 0:
    aggregate_stats = {
        "num_reports_processed": int(len(master_summary_df)),
        "avg_entity_coverage": float(master_summary_df["entity_coverage"].mean()),
        "avg_flow_coverage": float(master_summary_df["flow_coverage"].mean()),
        "avg_conditional_coverage": float(master_summary_df["conditional_coverage"].mean()),
        "avg_semantic_coverage": float(master_summary_df["semantic_coverage"].mean())
    }

    print(json.dumps(aggregate_stats, indent=2))

    with open(
        os.path.join(MASTER_OUTPUT_DIR, "aggregate_semantic_coverage_stats.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(aggregate_stats, f, indent=2)
else:
    print("No reports processed.")

# =========================
# Cell 20 — Final report-wise plot
# =========================

if len(master_summary_df) > 0:
    import matplotlib.pyplot as plt

    plot_df = master_summary_df.sort_values("report_folder").reset_index(drop=True)

    plt.figure(figsize=(16, 6))
    plt.bar(plot_df["report_folder"], plot_df["semantic_coverage"])
    plt.xticks(rotation=90)
    plt.ylabel("Semantic Coverage Score")
    plt.title("Report-wise Semantic Coverage")
    plt.tight_layout()
    plt.show()

# =========================
# Cell 21 — Optional: combine detailed files across all reports
# =========================

combined_entity_rows = []
combined_flow_rows = []
combined_conditional_rows = []

for report_dir in report_dirs:
    report_name = os.path.basename(report_dir)
    output_dir = os.path.join(report_dir, OUTPUT_DIR_NAME)

    entity_path = os.path.join(output_dir, "entity_coverage_details.csv")
    flow_path = os.path.join(output_dir, "flow_coverage_details.csv")
    conditional_path = os.path.join(output_dir, "conditional_coverage_details.csv")

    if os.path.exists(entity_path):
        temp_df = pd.read_csv(entity_path)
        temp_df.insert(0, "report_folder", report_name)
        combined_entity_rows.append(temp_df)

    if os.path.exists(flow_path):
        temp_df = pd.read_csv(flow_path)
        temp_df.insert(0, "report_folder", report_name)
        combined_flow_rows.append(temp_df)

    if os.path.exists(conditional_path):
        temp_df = pd.read_csv(conditional_path)
        temp_df.insert(0, "report_folder", report_name)
        combined_conditional_rows.append(temp_df)

if len(combined_entity_rows) > 0:
    pd.concat(combined_entity_rows, ignore_index=True).to_csv(
        os.path.join(MASTER_OUTPUT_DIR, "all_reports_entity_coverage_details.csv"),
        index=False
    )
    print("Saved combined entity coverage details.")

if len(combined_flow_rows) > 0:
    pd.concat(combined_flow_rows, ignore_index=True).to_csv(
        os.path.join(MASTER_OUTPUT_DIR, "all_reports_flow_coverage_details.csv"),
        index=False
    )
    print("Saved combined flow coverage details.")

if len(combined_conditional_rows) > 0:
    pd.concat(combined_conditional_rows, ignore_index=True).to_csv(
        os.path.join(MASTER_OUTPUT_DIR, "all_reports_conditional_coverage_details.csv"),
        index=False
    )
    print("Saved combined conditional coverage details.")